[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day19-vllm-pagedattention.ipynb)

# Day 19 — vLLM + PagedAttention

Today's big idea: the KV cache is **virtual memory**. vLLM's PagedAttention (Kwon et al., SOSP 2023) chops each request's KV cache into fixed-size blocks and maps logical positions to physical blocks through a per-request block table — exactly like an OS maps virtual pages to physical frames. The payoffs: no more 16 GiB contiguous preallocation, ~1.6% fragmentation instead of ~9x waste, prefix sharing via copy-on-write, and safe preemption under memory pressure.

**Objectives.** By the end of this lab you can:

1. Explain PagedAttention's block table as an OS page table, with numbers for Llama-3.1-8B.
2. Compute fragmentation waste: naive contiguous vs paged, in GiB and as a ratio.
3. Simulate block allocation, append, free, and copy-on-write prefix sharing in pure Python.
4. Simulate preemption: admit requests until the block budget runs out, evict the newest, recompute on return.
5. (Optional, T4) Serve a real model with vLLM and generate text.

In [ ]:
# No installs needed for the simulation: the lab below is pure Python (math, random).
# The optional vLLM demo in section 5 runs only on a T4 and installs vLLM there,
# guarded by torch.cuda.is_available() -- nothing to do on CPU.
print("CPU lab ready -- no installs needed")

# Expected output:
# CPU lab ready -- no installs needed

## 1. The big idea: the KV cache as virtual memory

A traditional engine reserves one **contiguous** KV slab per request sized for the *maximum* sequence length (e.g. 4096 tokens). Most requests use a fraction of that — memory fragmentation, exactly like external fragmentation in a heap allocator.

PagedAttention borrows the OS answer: split memory into fixed-size **blocks** (16 tokens each), keep a per-request **block table** (logical block → physical block), and allocate blocks on demand. Three OS tricks come along for free:

- **Demand paging** — a request only occupies blocks for the tokens it has actually generated.
- **Shared pages** — two requests with the same system prompt share the prefix's physical blocks, with copy-on-write when one of them diverges.
- **Preemption** — under pressure, evict a request's blocks (like swapping out a process) and recompute its KV when it is re-admitted.

## 2. KV geometry: how big is one block?

Llama-3.1-8B: 32 layers, 8 KV heads (GQA), head dim 128, fp16. Per token: 2 (K+V) × 32 layers × 8 heads × 128 dim × 2 bytes = **128 KiB/token**. A 16-token block is therefore 2 MiB.

In [ ]:
import math

# --- KV geometry: Llama-3.1-8B ---
N_LAYERS, N_KV_HEADS, HEAD_DIM, DTYPE_BYTES = 32, 8, 128, 2
BLOCK = 16  # PagedAttention block size, in tokens

def kv_bytes(n_layers, n_kv_heads, head_dim, seq_len, dtype_bytes=2):
    """KV cache bytes for seq_len tokens: K + V per layer per head."""
    return 2 * n_layers * n_kv_heads * head_dim * dtype_bytes * seq_len

TOK = kv_bytes(N_LAYERS, N_KV_HEADS, HEAD_DIM, 1)
BLOCK_BYTES = TOK * BLOCK
print(f"per-token KV : {TOK:,} B = {TOK/1024:.0f} KiB")
print(f"per-block KV : {BLOCK_BYTES:,} B = {BLOCK_BYTES/2**20:.0f} MiB  (block = {BLOCK} tokens)")

# Expected output:
# per-token KV : 131,072 B = 128 KiB
# per-block KV : 2,097,152 B = 2 MiB  (block = 16 tokens)

## 3. The simulator: physical blocks + per-request block tables

One pool of physical blocks, a refcount per block, and one block table per request. `alloc` reserves `ceil(len/16)` blocks; `append` grows the table on demand and copy-on-writes when the written block is shared; `free_req` returns blocks to the pool.

In [ ]:
class PagedKVCache:
    """PagedAttention KV cache: a pool of fixed-size physical blocks plus one
    block table per request (logical block id -> physical block id)."""
    def __init__(self, n_phys_blocks):
        self.free = list(range(n_phys_blocks))  # free physical block ids
        self.refcount = {}                       # phys id -> live references
        self.tables = {}                         # req -> [phys id per logical block]

    def _new(self):
        p = self.free.pop()
        self.refcount[p] = 1
        return p

    def alloc(self, req, n_tokens):
        """Reserve ceil(n_tokens / BLOCK) physical blocks for req."""
        tbl = self.tables.setdefault(req, [])
        for _ in range(math.ceil(n_tokens / BLOCK)):
            tbl.append(self._new())
        return tbl

    def append(self, req, n_tokens, seqlen):
        """Append tokens; copy-on-write when the written block is shared."""
        end = seqlen[req] + n_tokens
        tbl = self.tables[req]
        while len(tbl) < math.ceil(end / BLOCK):
            tbl.append(self._new())
        last = math.ceil(end / BLOCK) - 1
        p = tbl[last]
        if self.refcount.get(p, 1) > 1:
            q = self._new()
            self.refcount[p] -= 1
            tbl[last] = q
            print(f"  CoW: logical block {last} was phys {p} (refcount 2, shared) "
                  f"-> copied to phys {q}; {req} now points at {q}, refcount({p})=1")
        seqlen[req] = end

    def free_req(self, req):
        """Return req's physical blocks to the pool (table kept by the caller)."""
        for p in self.tables.pop(req, []):
            self.refcount[p] -= 1
            if self.refcount[p] == 0:
                del self.refcount[p]
                self.free.append(p)

print("PagedKVCache ready: block tables map logical -> physical block ids")

# Expected output:
# PagedKVCache ready: block tables map logical -> physical block ids

## 4. Fragmentation: contiguous slabs vs paged blocks

32 requests, lengths uniform U(100, 900) (seeded). Naive: every request preallocates a contiguous 4096-token slab. Paged: every request gets `ceil(len/16)` blocks — only the last block per request is partially filled, so internal fragmentation stays tiny.

In [ ]:
import random
random.seed(19)
lengths = [random.randint(100, 900) for _ in range(32)]
print(f"32 requests, mean length {sum(lengths)/len(lengths):.0f} tokens")

# Naive: every request preallocates a contiguous max-length (4096) slab
naive = 32 * kv_bytes(N_LAYERS, N_KV_HEADS, HEAD_DIM, 4096)

# Paged: every request gets ceil(len/16) blocks; only the last block is partial
used_tokens = sum(lengths)
alloc_tokens = sum(math.ceil(L / BLOCK) * BLOCK for L in lengths)
paged = alloc_tokens * TOK

print(f"naive contiguous : {naive/2**30:5.2f} GiB  (32 x 4096 tokens x 128 KiB)")
print(f"paged blocks     : {paged/2**30:5.2f} GiB  ({alloc_tokens:,} block slots for {used_tokens:,} live tokens)")
print(f"memory saving    : {naive/paged:.2f}x  -- the paged run needs ~1/9th the GPU memory")
print(f"internal frag    : {(paged - used_tokens*TOK)/paged:.2%}  (only each request's last block is partially filled)")

# Expected output:
# 32 requests, mean length 449 tokens
# naive contiguous : 16.00 GiB  (32 x 4096 tokens x 128 KiB)
# paged blocks     :  1.78 GiB  (14,592 block slots for 14,355 live tokens)
# memory saving    :  8.98x  -- the paged run needs ~1/9th the GPU memory
# internal frag    :  1.62%  (only each request's last block is partially filled)

## 5. Copy-on-write prefix sharing (prefix caching for free)

Two requests share a 100-token system prompt — one physical copy, refcount 2, both block tables point at the same blocks. When request A appends into the shared last block, CoW copies just that block instead of duplicating the whole prefix.

In [ ]:
cache = PagedKVCache(64)
seqlen = {}
# The 100-token shared prefix: 7 blocks, allocated ONCE, referenced by both requests
prefix = [cache._new() for _ in range(math.ceil(100 / BLOCK))]
for p in prefix:
    cache.refcount[p] = 2
cache.tables["A"] = list(prefix)
cache.tables["B"] = list(prefix)
seqlen["A"] = seqlen["B"] = 100

print("before: A table =", cache.tables["A"])
print("before: B table =", cache.tables["B"])
print("refcounts:", sorted(cache.refcount.items()))

# A appends 3 tokens (100..102) into the shared last block -> CoW
cache.append("A", 3, seqlen)

print("after:  A table =", cache.tables["A"])
print("after:  B table =", cache.tables["B"])
dup = 2 * len(prefix) * BLOCK_BYTES / 2**20
shared = (len(prefix) + 1) * BLOCK_BYTES / 2**20
print(f"two full prefix copies: {dup:.0f} MiB  vs  shared + 1 CoW block: {shared:.0f} MiB"
      f"  -> saved {dup - shared:.0f} MiB")
print("phys blocks free:", len(cache.free))

# Expected output:
# before: A table = [63, 62, 61, 60, 59, 58, 57]
# before: B table = [63, 62, 61, 60, 59, 58, 57]
# refcounts: [(57, 2), (58, 2), (59, 2), (60, 2), (61, 2), (62, 2), (63, 2)]
#   CoW: logical block 6 was phys 57 (refcount 2, shared) -> copied to phys 56; A now points at 56, refcount(57)=1
# after:  A table = [63, 62, 61, 60, 59, 58, 56]
# after:  B table = [63, 62, 61, 60, 59, 58, 57]
# two full prefix copies: 28 MiB  vs  shared + 1 CoW block: 16 MiB  -> saved 12 MiB
# phys blocks free: 56

## 6. Eviction: admit until the block budget is full

Fixed pool of 10 physical blocks. Admit r0 (80 tok) and r1 (48 tok); when r2 (96 tok, 6 blocks) arrives and does not fit, preempt newest-first: physical blocks are freed, the block table is kept, and the evicted requests' KV is recomputed when they are re-admitted.

In [ ]:
ev = PagedKVCache(10)
seq2, evicted = {}, []

for req, L in [("r0", 80), ("r1", 48)]:
    need = math.ceil(L / BLOCK)
    ev.alloc(req, L); seq2[req] = L
    print(f"admitted {req} ({L} tok, {need} blocks) -> free blocks: {len(ev.free)}")

req, L = "r2", 96
need = math.ceil(L / BLOCK)
print(f"{req} arrives needing {need} blocks, free={len(ev.free)}: insufficient -> preempting")
for victim in reversed(list(ev.tables)):   # newest first, like vLLM
    if len(ev.free) >= need:
        break
    ev.free_req(victim); evicted.append(victim)
    print(f"  preempted {victim}: freed its blocks (table kept for recompute), free={len(ev.free)}")
ev.alloc(req, L); seq2[req] = L
print(f"admitted {req} ({L} tok, {need} blocks) -> free blocks: {len(ev.free)}")
print("preempted this round:", evicted, "(their tables survive -> recompute on return)")

# Expected output:
# admitted r0 (80 tok, 5 blocks) -> free blocks: 5
# admitted r1 (48 tok, 3 blocks) -> free blocks: 2
# r2 arrives needing 6 blocks, free=2: insufficient -> preempting
#   preempted r1: freed its blocks (table kept for recompute), free=5
#   preempted r0: freed its blocks (table kept for recompute), free=10
# admitted r2 (96 tok, 6 blocks) -> free blocks: 4
# preempted this round: ['r1', 'r0'] (their tables survive -> recompute on return)

## 7. Optional: the real thing on a T4

The simulation above is the full lab on CPU. On a free T4 runtime the cell below installs vLLM, serves SmolLM2-360M (~0.7 GB, fp16, `max_model_len=1024` — fits comfortably), and generates a one-sentence explanation of PagedAttention. Runtime: a few minutes for the install + weights.

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
print("cuda available:", HAS_GPU)

if not HAS_GPU:
    print("No GPU -- the CPU simulation above is the full lab;")
    print("switch to a T4 runtime to run the vLLM branch for real.")
else:
    !pip install -q vllm
    from vllm import LLM, SamplingParams
    llm = LLM(model="HuggingFaceTB/SmolLM2-360M", max_model_len=1024, dtype="float16")
    outs = llm.generate(["Explain PagedAttention in one sentence."],
                        SamplingParams(max_tokens=40, temperature=0))
    print(outs[0].outputs[0].text)

# Expected output (CPU path):
# cuda available: False
# No GPU -- the CPU simulation above is the full lab;
# switch to a T4 runtime to run the vLLM branch for real.
#
# Expected output (T4 path): one-sentence explanation of PagedAttention, e.g.
# "PagedAttention stores the KV cache in fixed-size blocks, like virtual-memory pages, ..."

## What to measure

Record the day's numbers here (from your run, not from memory):

| Metric | Your number |
|---|---|
| Naive KV, 32 reqs (GiB) | 16.00 |
| Paged KV, 32 reqs (GiB) | 1.78 |
| Memory saving ratio | 8.98x |
| Internal fragmentation | 1.62% |
| CoW bytes saved (100-tok shared prefix) | 12 MiB |
| Requests preempted under 10-block budget | r1, r0 (newest first) |

## Checkpoints

**1. Why does contiguous KV preallocation waste ~9x memory on the 32-request workload?**

   Every request reserves a full 4096-token slab but uses ~449 tokens on average; paged allocation only pays for `ceil(len/16)` blocks, so memory tracks actual tokens, not the max.

**2. What does a block table map, and what does refcount > 1 on a physical block mean?**

   Logical block id → physical block id. Refcount > 1 means the block is shared (e.g. a common system prompt), so a write into it triggers copy-on-write instead of corrupting the other request.

**3. On preemption, what is freed, what is kept, and what happens when the request returns?**

   The physical blocks are freed to the pool; the block table (the logical layout) is kept, and the request's KV is recomputed from its tokens when it is re-admitted.

## Tomorrow

**Day 20 — vLLM scheduling:** FCFS with preemption, and chunked prefill — the scheduler that sits on top of today's block allocator. You will simulate the vLLM scheduling loop and measure how chunked prefill keeps decode latency flat under mixed prompt lengths.